# Pelajaran 18: Mengamankan Agen AI dengan Resi Kriptografi

## Notebook Praktik

Notebook ini menjelaskan empat latihan:

1. **Tandatangani resi pertama Anda** untuk panggilan alat agen dan verifikasi.
2. **Ganggu resi** dan lihat verifikasi gagal.
3. **Buat rantai tiga resi** dan konfirmasi integritas rantai.
4. **Bungkus panggilan alat Microsoft Agent Framework** sehingga setiap aksi menghasilkan resi.

Semua primitif kriptografi diimpor dari perpustakaan yang terawat dengan baik (`pynacl` untuk Ed25519, `jcs` untuk JSON kanonik RFC 8785, `hashlib` dari pustaka standar Python untuk SHA-256). Logika resi itu sendiri adalah Python biasa yang dapat Anda baca dan modifikasi.

Jalankan sel-sel secara berurutan. Setiap bagian singkat dan mandiri.


## Setup

Instal dua dependensi tersebut. Keduanya memiliki lisensi permisif (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Utilitas Pembantu

Dua pembantu ini menangani pengkodean base64url (tanpa padding) dan hashing SHA-256 dari objek arbitrer. Mereka menjaga agar sisa notebook tetap fokus pada logika tanda terima itu sendiri.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Bagian 1: Tandatangani tanda terima pertama Anda

Bayangkan agen kami untuk **Contoso Travel** baru saja mencari penerbangan dari Sydney ke Los Angeles untuk seorang pelanggan. Kami ingin merekam pernyataan gateway tentang panggilan alat ini sehingga auditor di masa depan dapat memverifikasinya di bawah kunci penerbit yang dipasang secara terpisah.

### Langkah 1.1: Buat kunci tanda tangan

Dalam produksi, kunci tanda tangan agen akan disimpan dalam modul keamanan perangkat keras (HSM), Azure Key Vault, atau penyimpanan terlindungi serupa. Untuk pelajaran ini kami membuat kunci baru dalam memori.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Langkah 1.2: Bangun payload tanda terima

Payload berisi semua yang ingin kita buktikan dalam tanda terima: siapa yang bertindak, alat apa yang digunakan, dengan argumen apa, apa yang dikembalikan, di bawah kebijakan apa, dan kapan. Kita melakukan hash pada argumen dan hasilnya daripada memasukkannya secara langsung agar tanda terima tidak membocorkan konten yang sensitif.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Langkah 1.3: Tandatangani dan rangkai tanda terima

Tiga langkah:

1. Kanonisasikan payload menggunakan JCS sehingga dua implementasi yang menghasilkan tanda terima logis yang sama menghasilkan byte yang identik secara byte.
2. Tandatangani byte kanonis secara langsung dengan kunci privat Ed25519. PureEdDSA melakukan hash pesan secara internal, jadi prahash tambahan akan mengubah protokol.

Tanda tangan kemudian dilampirkan pada payload asli untuk menghasilkan tanda terima akhir.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Langkah 1.4: Verifikasi tanda terima

Verifikasi membalikkan prosesnya. Kami menghapus tanda tangan, menghitung ulang byte kanonik, dan memeriksa tanda tangan terhadap kunci publik penerbit.

Verifikasi dilakukan secara offline: tidak ada layanan yang dipanggil dan tidak ada akses jaringan. Namun masih membutuhkan satu hal selain tanda terima: kunci publik penerbit, yang diperoleh melalui saluran yang sudah dipercaya auditor, seperti JWK Set yang diterbitkan oleh penerbit atau nilai yang telah dipasang sebelumnya. Field `public_key` di dalam tanda terima tidak dapat memainkan peran itu, karena siapa pun yang menulis tanda terima juga menulis field tersebut. Bagian 2 menunjukkan pemalsuan yang dicegah oleh hal ini.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Anda harus melihat `Receipt is valid: True`, `Pre-hashed receipt valid: False` dan `Malformed receipts rejected: True`. Kasus positif membuktikan jalur penandatanganan direct-JCS berfungsi; kontrol negatif membuat aturan ruang lingkup tanda tangan dapat dieksekusi alih-alih hanya sebagai prosa; baris terakhir menunjukkan input yang salah format mengembalikan `False` alih-alih memunculkan pengecualian.


## Bagian 2: Memanipulasi struk

Tujuan utama dari struk adalah agar dapat diketahui jika telah dimanipulasi. Mari buktikan.

Kami akan mengubah tepat satu karakter dari struk dan melihat verifikasi gagal.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Apa yang baru saja terjadi?

Ketika kami mengubah `policy_id`, byte kanonik berubah. Tanda tangan (yang sebelumnya berlaku pada byte kanonik asli) tidak lagi cocok. Verifikasi dengan benar mengembalikan `False`.

Tidak ada cara untuk mengubah bidang yang ditandatangani, yaitu apapun di luar objek `signature`, dan tetap membuat tanda terima diverifikasi dengan kunci publik penerbit, kecuali penyerang memiliki kunci privat penerbit. Objek `signature` itu sendiri tidak ditandatangani: `sig` adalah tanda tangan, dan `verify_receipt` mengabaikan `public_key`, jadi mengedit bidang itu saja tidak mengubah apa-apa. Selama kunci privat berada di brankas kunci dan pemeriksa menggunakan kunci penerbit yang dipublikasikan secara independen, perubahan pada bidang yang ditandatangani tidak mungkin disembunyikan. Kondisi itu penting: sel berikutnya menunjukkan tanda terima yang dimodifikasi yang diverifikasi dengan sempurna terhadap kunci yang berbeda.

Coba sendiri: ubah `tool_name` atau `agent_id` atau `timestamp` di sel di atas dan jalankan ulang. Setiap perubahan pada bidang yang ditandatangani menghasilkan tanda terima yang tidak valid.


### Pemalsuan yang tidak tertangkap oleh tanda tangan

Penyerang di atas tidak menandatangani ulang. Yang lebih hati-hati akan: mengubah tanda terima,
menandatanganinya dengan kunci yang mereka buat sendiri, dan menaruh kunci itu di
bidang `public_key`. Hasilnya adalah tanda terima yang tanda tangannya benar-benar asli
untuk isinya.

Jalankan ini dan perhatikan pemeriksaan mana dari kedua yang menangkapnya.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Anda harus melihat `True` lalu `False`. Tiga baris `assert` membuat itu menjadi
pemeriksaan: sel menghasilkan error jika pemalsuan tersebut pernah diverifikasi terhadap
kunci penerbit.

Pemalsuan itu secara internal konsisten. Tandatangan nya benar-benar cocok dengan
isinya dan kunci yang dibawanya, jadi pemeriksa yang membaca kunci dari
tanda terima melaporkannya sebagai valid. Tidak ada yang salah dengan tanda terima,
karena tidak ada yang tidak konsisten di dalamnya.

Satu-satunya hal yang menangkapnya adalah pemeriksaan terhadap kunci yang tidak bisa
dipengaruhi oleh penyerang. Ini sebabnya `verify_receipt` mengambil kunci sebagai argumen:
sebuah fungsi dengan tanda tangan `verify_receipt(receipt)` tidak dapat ditulis dengan aman,
karena satu-satunya kunci yang bisa dicapai adalah yang disediakan oleh penyerang.

Tanda terima yang sama disimpan sebagai `sample_receipts/04_resigned_receipt.json` jika Anda
ingin memeriksanya di luar notebook.


## Bagian 3: Rangkai tanda terima bersama-sama

Satu tanda terima melindungi satu tindakan. Sebagian besar agen melakukan banyak tindakan. Untuk membuat seluruh urutan menjadi terindikasi perubahan, kami menghubungkan setiap tanda terima dengan tanda terima sebelumnya dengan memasukkan hash tanda terima sebelumnya dalam payload tanda terima baru.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jika ada yang menghapus atau mengubah urutan tanda terima di dalam urutan yang disajikan, tautan berikutnya akan terputus. Mendeteksi pemotongan atau penulisan ulang sufiks yang dikontrol penandatangan juga memerlukan kepala saat ini yang dipaku, titik pemeriksaan eksternal, atau komitmen populasi yang terautentikasi.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Sekarang putus rantainya dengan merusak tanda terima tengah dan verifikasi ulang. Tanda terima yang dirusak gagal dalam pemeriksaan tanda tangan, DAN tanda terima berikutnya gagal dalam pemeriksaan tautan rantai (karena `previous_receipt_hash` nya tidak lagi cocok dengan hash tanda terima tengah yang dimodifikasi).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Tanda terima 0 masih diverifikasi (tidak diubah dan tidak memiliki penerus untuk bergantung). Tanda terima 1 gagal pemeriksaan tanda tangannya karena kami mengubah `tool_args_hash`. Tanda terima 2 gagal pemeriksaan rantai-tautannya karena `previous_receipt_hash` dihitung terhadap tanda terima 1 yang asli (yang sekarang dimodifikasi).

Bahkan jika penyerang menandatangani ulang tanda terima 1 yang dimodifikasi, tanda tangan baru gagal terhadap kunci penerbit kecuali mereka memegang kunci pribadi penerbit (penandatanganan ulang dengan kunci mereka sendiri gagal, seperti yang ditunjukkan Bagian 2), dan ketidakcocokan rantai-tautan di tanda terima 2 tetap akan memperlihatkan pemalsuan. Untuk menyembunyikan perubahan, penyerang harus menandatangani ulang setiap tanda terima dari titik perubahan dan seterusnya, yang memerlukan kepemilikan kunci pribadi penerbit.


## Bagian 4: Membungkus pemanggilan alat agen dengan penandatanganan tanda terima

Dalam penerapan nyata, Anda tidak ingin setiap penulis agen harus ingat untuk memanggil `make_receipt`. Anda ingin penandatanganan tanda terima otomatis untuk setiap pemanggilan alat.

Berikut pola paling sederhana: kelas pembungkus yang mengambil fungsi alat yang dapat dipanggil dan mengembalikan versi yang mengeluarkan tanda terima. Ini beradaptasi dengan kerangka kerja agen apa pun, termasuk Microsoft Agent Framework (`agent_framework.foundry`).

Jika Anda belum mengatur proyek Microsoft Foundry, contoh lokal palsu di bawah ini tetap menunjukkan pola tersebut.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integrasi dengan Microsoft Agent Framework

Pembungkus `ReceiptedTool` di atas tidak tergantung pada framework. Untuk menggunakannya di dalam agen yang dibangun dengan Microsoft Agent Framework, daftarkan fungsi yang dibungkus tersebut sebagai alat. Sebuah sketsa (anda akan mengganti tiruan dengan pendaftaran alat Microsoft Foundry yang nyata):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Framework agen tidak perlu mengetahui apa pun tentang kwitansi. Penandatanganan kwitansi dibungkus di sekitar alat, bukan dipasangkan ke dalam framework. Beginilah cara menambahkan asal-usul ke kode agen yang sudah ada tanpa menulis ulang agen.


## Recap and stretch challenge

You have:

- Generated an Ed25519 key pair.
- Built and signed a receipt for an agent tool call.
- Verified the receipt offline using only the issuer's public key, obtained independently of the receipt.
- Tampered with a receipt and observed verification fail.
- Built a hash-chained sequence of three receipts.
- Tampered with the middle of the chain and observed both signature failure and chain-link failure.
- Wrapped a tool function with automatic receipt signing.

**Stretch challenge.** Extend the receipt schema with a `request_id` field (a UUID for distributed tracing). Update `make_receipt` to include it, and confirm receipts still verify end to end. Then modify the field after signing and confirm verification fails. This forces you to internalize how every byte of the canonical encoding contributes to the signature.

**Important boundary.** Under a pinned key and chain context, receipts prove signature attribution, content integrity, and a link to the presented predecessor. They do NOT prove execution, source truth, trusted time, completeness, policy evaluation, or correctness. Receipts are a foundation. Governance is the system you build on top.

Read the lesson README again with that boundary in mind. The most common mistake teams make with receipts is assuming "we have receipts" means "we are governed." It does not. Receipts make agent behavior auditable. They do not make it correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
